[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DigitalAgroecosystemsLab/AGRI3100-Practicals-2026/blob/main/AGRI3100_Lab2_Agricultural_Data_Handling_Visualization.ipynb)


# AGRI 3100 – Introduction to Digital Agriculture
## Lab 2 – Agricultural Data Handling and Visualization in Digital Agriculture

### Purpose
In this practical class, you will work with a real soil dataset hosted in Google Drive. You will use **Python in Google Colab** to:

- connect to and load agricultural data;
- inspect the structure of a large dataset;
- identify missing values and data-quality issues;
- summarize important soil properties;
- visualize agricultural data;
- explore relationships among soil properties; and
- plot soil spectral reflectance signatures from **350–2500 nm**.

### Dataset
The workbook contains several worksheets. In this lab, we will mainly use **`Matched Data`**, which combines soil laboratory measurements with spectral reflectance data.

> **Important:** Do not edit the master dataset. Your Colab session works with a temporary copy.

## Before you start

> **Important:** This notebook is running in a temporary Colab session. Periodically save a backup using **File → Download → Download .ipynb**. When you finish the lab, download the completed `.ipynb` file and submit it through UM Learn. You do not need to save this notebook to Google Drive.


## Part 0 – Set up the Colab environment

Run the cell below once. It installs the small package used to retrieve the shared dataset and imports the Python libraries used in this lab.

In [ ]:
!pip -q install gdown openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("✓ Python environment is ready.")

## Part 1 – Connect to the shared AGRI 3100 dataset

The dataset is stored in the instructor's Google Drive with **view-only** access.

The code below downloads a temporary copy into your Colab session. You do not need to upload the file yourself.

In [ ]:
import gdown
import os

FILE_ID = "1s6efVgTs8YLwwcxqaAcfLtbc4BYyobOp"
FILE_NAME = "AGRI3100-Soil-Data.xlsx"

if not os.path.exists(FILE_NAME):
    gdown.download(
        id=FILE_ID,
        output=FILE_NAME,
        quiet=False
    )

print(f"✓ Dataset available as: {FILE_NAME}")

### Check the workbook structure

Before analyzing data, identify the worksheets available in the Excel workbook.

In [ ]:
excel_file = pd.ExcelFile(FILE_NAME)

print("Worksheets in the workbook:")
for sheet in excel_file.sheet_names:
    print(" •", sheet)

### Load the main worksheet

For this lab, we will use the **`Matched Data`** worksheet.

Each row represents a soil sample. The columns include sample information, laboratory soil properties, quality-control information, and spectral reflectance measurements.

In [ ]:
df = pd.read_excel(FILE_NAME, sheet_name="Matched Data")

print("✓ Matched Data loaded successfully")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

## Part 2 – Understand the dataset

A large digital-agriculture dataset can contain many different types of variables.

The first columns in this dataset contain:
- sample identifiers;
- sample depth;
- soil pH;
- total organic carbon (TOC);
- particle-size measurements;
- sand, silt, and clay;
- soil texture;
- calcium carbonate equivalent (CCE);
- electrical conductivity (EC); and
- spectral quality-control information.

The remaining columns contain spectral reflectance measurements at wavelengths from **350 to 2500 nm**.

In [ ]:
print("First 30 column names:")
for i, col in enumerate(df.columns[:30], start=1):
    print(f"{i:>2}. {col}")

### Separate soil-property columns from spectral columns

This makes the large dataset easier to work with.

In [ ]:
soil_columns = [
    "Sample_ID",
    "Source_Year",
    "Depth",
    "1:2 pH",
    "TOC (%)",
    "Sand (%)",
    "Silt (%)",
    "Clay (%)",
    "Texture / Pipette",
    "CCE (%)",
    "EC (mmhos/cm)"
]

# Spectral columns are wavelength labels from 350 through 2500 nm.
spectral_columns = [
    col for col in df.columns
    if str(col).isdigit() and 350 <= int(col) <= 2500
]

soil = df[soil_columns].copy()

print(f"Soil-property table: {soil.shape[0]:,} rows × {soil.shape[1]} columns")
print(f"Spectral variables: {len(spectral_columns):,} wavelengths")
print(f"Spectral range: {spectral_columns[0]}–{spectral_columns[-1]} nm")

soil.head()

### Student Checkpoint 1

Answer these questions in your lab notes:

1. How many soil samples are in the `Matched Data` worksheet?
2. How many total columns are in the worksheet?
3. How many wavelength variables are present?
4. What is the wavelength range?
5. In your own words, what does one **row** represent?

## Part 3 – Inspect data types and missing values

Real agricultural datasets are rarely perfect. Before calculating statistics or making graphs, we should inspect the data.

In [ ]:
soil.info()

### Count missing observations

In [ ]:
missing = soil.isna().sum().sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Missing values": missing,
    "Percent missing": (missing / len(soil) * 100).round(2)
})

missing_table

### Visualize missing values

This graph shows the percentage of missing observations for each selected soil variable.

In [ ]:
missing_plot = missing_table[missing_table["Percent missing"] > 0].sort_values(
    "Percent missing",
    ascending=True
)

plt.figure(figsize=(9, 5))
plt.barh(missing_plot.index, missing_plot["Percent missing"])
plt.xlabel("Missing observations (%)")
plt.ylabel("Variable")
plt.title("Missing Data in Selected Soil Variables")
plt.tight_layout()
plt.show()

### Check duplicate sample identifiers

In [ ]:
duplicate_ids = df["Sample_ID"].duplicated().sum()

print(f"Duplicate Sample_ID values: {duplicate_ids:,}")

### Check whether sand + silt + clay approximately equals 100%

Particle-size fractions should normally sum to approximately 100%. Small differences may result from rounding or laboratory procedures.

In [ ]:
soil["Sand (%)"] = pd.to_numeric(soil["Sand (%)"], errors="coerce")
soil["Clay (%)"] = pd.to_numeric(soil["Clay (%)"], errors="coerce")

texture_check = soil[["Sample_ID", "Sand (%)", "Silt (%)", "Clay (%)"]].copy()

texture_check["Texture Sum (%)"] = (
    texture_check["Sand (%)"] +
    texture_check["Silt (%)"] +
    texture_check["Clay (%)"]
)

texture_check["Difference from 100"] = texture_check["Texture Sum (%)"] - 100

texture_check.head(10)

In [ ]:
print("Summary of sand + silt + clay:")
print(texture_check["Texture Sum (%)"].describe())

outside_tolerance = texture_check[
    texture_check["Texture Sum (%)"].notna() &
    ((texture_check["Texture Sum (%)"] < 98) |
     (texture_check["Texture Sum (%)"] > 102))
]

print(f"\nSamples outside 98–102%: {len(outside_tolerance):,}")

### Student Checkpoint 2 – Data quality

Write short answers:

1. Which selected soil variable has the most missing observations?
2. Are there duplicate `Sample_ID` values?
3. Do sand + silt + clay always equal exactly 100%?
4. Give **two reasons** why checking data quality is important before analysis.

## Part 4 – Prepare numeric soil variables

Sometimes columns imported from spreadsheets contain text, blanks, or unusual values. We will explicitly convert the soil measurements used in this lab to numeric values.

Values that cannot be interpreted as numbers will become `NaN` (missing values).

In [ ]:
numeric_soil_vars = [
    "1:2 pH",
    "TOC (%)",
    "Sand (%)",
    "Silt (%)",
    "Clay (%)",
    "CCE (%)",
    "EC (mmhos/cm)"
]

for col in numeric_soil_vars:
    soil[col] = pd.to_numeric(soil[col], errors="coerce")

print("✓ Selected soil variables converted to numeric form.")

## Part 5 – Descriptive statistics

Descriptive statistics provide a quick overview of the central tendency and variability of agricultural measurements.

In [ ]:
summary_stats = soil[numeric_soil_vars].describe().T

summary_stats = summary_stats[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)

summary_stats

### Compare mean and median

The median is the 50th percentile. Large differences between the mean and median can indicate a skewed distribution or extreme values.

In [ ]:
mean_median = pd.DataFrame({
    "Mean": soil[numeric_soil_vars].mean(),
    "Median": soil[numeric_soil_vars].median()
}).round(2)

mean_median["Mean - Median"] = (
    mean_median["Mean"] - mean_median["Median"]
).round(2)

mean_median

# Part 6 – Visualizing agricultural data

Good visualization helps us identify patterns, variability, outliers, and relationships that may not be obvious from tables alone.

We will create four core figures.

## Figure 1 – Distribution of soil pH

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(soil["1:2 pH"].dropna(), bins=20, edgecolor="black")
plt.xlabel("Soil pH (1:2)")
plt.ylabel("Number of samples")
plt.title("Distribution of Soil pH")
plt.tight_layout()
plt.show()

### Question
Describe the soil pH distribution in 2–3 sentences. Where are most samples concentrated? Are there unusually low or high values?

## Figure 2 – Distribution of Total Organic Carbon (TOC)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(soil["TOC (%)"].dropna(), bins=25, edgecolor="black")
plt.xlabel("Total Organic Carbon (%)")
plt.ylabel("Number of samples")
plt.title("Distribution of Soil Total Organic Carbon")
plt.tight_layout()
plt.show()

### Question
Compare the TOC distribution with the pH distribution. Which appears more skewed? What might cause a soil-property distribution to be skewed?

## Figure 3 – Number of samples by soil texture class

In [ ]:
texture_counts = (
    soil["Texture / Pipette"]
    .dropna()
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(9, 6))
plt.barh(texture_counts.index, texture_counts.values)
plt.xlabel("Number of soil samples")
plt.ylabel("Texture class")
plt.title("Samples by Soil Texture Class")
plt.tight_layout()
plt.show()

### Question
Which texture classes are most represented? Why should we care about unequal numbers of samples among classes?

## Figure 4 – Relationship between clay and TOC

Scatterplots are useful for investigating relationships between two continuous variables.

In [ ]:
plot_data = soil[["Clay (%)", "TOC (%)"]].dropna()

plt.figure(figsize=(8, 6))
plt.scatter(
    plot_data["Clay (%)"],
    plot_data["TOC (%)"],
    alpha=0.6
)
plt.xlabel("Clay (%)")
plt.ylabel("Total Organic Carbon (%)")
plt.title("Relationship Between Clay Content and Soil Organic Carbon")
plt.tight_layout()
plt.show()

### Calculate a correlation coefficient

The Pearson correlation coefficient ranges from **−1 to +1**.

- Positive values indicate that variables tend to increase together.
- Negative values indicate that one tends to decrease as the other increases.
- Values near zero indicate little linear relationship.

Correlation does **not** prove causation.

In [ ]:
clay_toc_corr = plot_data["Clay (%)"].corr(plot_data["TOC (%)"])

print(f"Pearson correlation between Clay (%) and TOC (%): {clay_toc_corr:.3f}")

### Question
Describe the clay–TOC relationship in your own words. Is the relationship strong, moderate, weak, positive, or negative? Does the plot contain possible outliers?

## Part 7 – Correlation among selected soil properties

A correlation matrix provides a compact view of relationships among several numeric variables.

In [ ]:
corr_vars = ["1:2 pH", "TOC (%)", "Sand (%)", "Silt (%)", "Clay (%)", "EC (mmhos/cm)"]
corr_matrix = soil[corr_vars].corr()

corr_matrix.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

image = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")

ax.set_xticks(range(len(corr_vars)))
ax.set_yticks(range(len(corr_vars)))
ax.set_xticklabels(corr_vars, rotation=45, ha="right")
ax.set_yticklabels(corr_vars)

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        value = corr_matrix.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center")

plt.colorbar(image, ax=ax, label="Pearson correlation")
plt.title("Correlation Matrix of Selected Soil Properties")
plt.tight_layout()
plt.show()

# Part 8 – Introduction to soil spectroscopy

The dataset contains reflectance measurements from **350 to 2500 nm**.

A sequence of reflectance values across wavelengths forms a **spectral signature**. Spectral signatures can contain information related to soil organic matter, minerals, moisture, texture, and other soil characteristics.

First, convert the wavelength columns to numeric data.

In [ ]:
spectra = df[spectral_columns].apply(pd.to_numeric, errors="coerce")

wavelengths = np.array([int(col) for col in spectral_columns])

print(f"Spectral matrix dimensions: {spectra.shape[0]:,} samples × {spectra.shape[1]:,} wavelengths")
print(f"Wavelength range: {wavelengths.min()}–{wavelengths.max()} nm")

## Plot spectral signatures for three soil samples

In [ ]:
# Select three samples that contain spectral data.
valid_spectra = spectra.dropna(how="all")

sample_indices = valid_spectra.index[:3]

plt.figure(figsize=(11, 6))

for idx in sample_indices:
    reflectance = spectra.loc[idx].to_numpy(dtype=float)
    sample_id = df.loc[idx, "Sample_ID"]

    plt.plot(
        wavelengths,
        reflectance,
        linewidth=1.5,
        label=f"Sample {sample_id}"
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.title("Soil Spectral Reflectance Signatures")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Question
Look at the three spectral curves.

1. Are the curves identical?
2. At approximately what wavelength regions do you see major changes in reflectance?
3. Why might different soil samples have different spectral signatures?
4. How could these spectra potentially be used in digital agriculture?

## Part 9 – Compare spectra for low- and high-TOC soils

Instead of choosing arbitrary samples, we can compare samples representing different soil conditions.

In [ ]:
spectral_toc = pd.concat(
    [soil[["Sample_ID", "TOC (%)"]], spectra],
    axis=1
).dropna(subset=["TOC (%)"])

# Keep samples that have at least some spectral data.
spectral_toc = spectral_toc.dropna(subset=spectral_columns, how="all")

low_idx = spectral_toc["TOC (%)"].idxmin()
high_idx = spectral_toc["TOC (%)"].idxmax()

plt.figure(figsize=(11, 6))

for idx, label in [
    (low_idx, "Lowest TOC"),
    (high_idx, "Highest TOC")
]:
    reflectance = spectra.loc[idx].to_numpy(dtype=float)

    plt.plot(
        wavelengths,
        reflectance,
        linewidth=1.5,
        label=(
            f"{label}: Sample {df.loc[idx, 'Sample_ID']} "
            f"(TOC = {soil.loc[idx, 'TOC (%)']:.2f}%)"
        )
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.title("Spectral Signatures of Low- and High-TOC Soil Samples")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Discussion
Do not conclude from only two samples that TOC alone caused the observed spectral differences. Soil spectra can also be affected by texture, mineralogy, moisture, sample preparation, measurement conditions, and other properties.

This illustrates an important principle in digital agriculture:

> **Sensor measurements must be interpreted together with field and laboratory information.**

# Part 10 – Independent Student Challenge

Complete the following task independently.

### Your task
Choose **one soil property** from:

- `1:2 pH`
- `TOC (%)`
- `Sand (%)`
- `Silt (%)`
- `Clay (%)`
- `CCE (%)`
- `EC (mmhos/cm)`

Then:

1. Calculate its mean, median, minimum, maximum, and number of missing observations.
2. Create **one appropriate visualization**.
3. Give the graph a meaningful title and label both axes.
4. Write **3–4 sentences** explaining what the visualization tells you about the soil dataset.

Use the code cell below as your workspace.

In [ ]:
# ============================================================
# INDEPENDENT CHALLENGE – STUDENT WORKSPACE
# ============================================================

chosen_variable = "Clay (%)"   # Change this to your selected variable.

print("Variable:", chosen_variable)
print("Mean:", soil[chosen_variable].mean())
print("Median:", soil[chosen_variable].median())
print("Minimum:", soil[chosen_variable].min())
print("Maximum:", soil[chosen_variable].max())
print("Missing:", soil[chosen_variable].isna().sum())

# Create your visualization below.
plt.figure(figsize=(8, 5))
plt.hist(soil[chosen_variable].dropna(), bins=20, edgecolor="black")

plt.xlabel(chosen_variable)
plt.ylabel("Number of samples")
plt.title(f"Distribution of {chosen_variable}")

plt.tight_layout()
plt.show()

# Write your interpretation in a Markdown cell below this cell.

# Part 11 – Final Reflection

Before finishing the lab, answer the following questions.

### Reflection Questions

1. Why is data inspection necessary before visualization or modelling?
2. What is the difference between a categorical variable and a numerical variable? Give one example of each from this dataset.
3. Why can missing data be a problem?
4. What information can a histogram provide that a mean alone cannot?
5. What is the purpose of a scatterplot?
6. Does correlation demonstrate causation? Explain briefly.
7. What is a soil spectral signature?
8. Name two soil characteristics that could influence spectral reflectance.
9. In one or two sentences, explain how this lab relates to **digital agriculture**.

# Lab 2 – Completion Checklist

Before submitting your work, confirm that you have:

- [ ] successfully connected to the shared dataset;
- [ ] loaded the `Matched Data` worksheet;
- [ ] examined rows, columns, and variable types;
- [ ] evaluated missing data and basic data-quality issues;
- [ ] calculated descriptive statistics;
- [ ] produced the four required soil-data figures;
- [ ] examined the correlation matrix;
- [ ] plotted soil spectral signatures;
- [ ] completed the independent visualization challenge; and
- [ ] answered the checkpoint and reflection questions.

### Saving your work

> **Important:** This notebook is running in a temporary Colab session. Periodically save a backup using **File → Download → Download .ipynb**. When you finish the lab, download the completed `.ipynb` file and submit it through UM Learn. You do not need to save this notebook to Google Drive.

Rename your downloaded file using the format:

`AGRI3100_Lab2_LastName_FirstName.ipynb`

Follow the instructor's submission instructions in UM Learn.
